# FDHN — Improved Implementation (v2)

Builds on the base FDHN implementation with training-level improvements aimed at closing the gap to the paper's full-LIAR2 result (0.702 accuracy):

- **AdamW + weight decay** (L2 regularization)
- **Class-weighted loss** (targets weak recall on `pants-fire` / `mostly-true`)
- **Gradient clipping**
- **ReduceLROnPlateau** learning-rate scheduler
- **Early stopping** based on validation accuracy

Same base architecture as before (TextCNN x2, CNNBiLSTM, Gaussian Fuzzy Layer, justification branch, Fuse Layer) — only the *training* is improved here, so you can directly compare against your base run's 0.548 accuracy / 0.519 F1-macro.

**Setup:** place this notebook in the same folder as your `liar2/` data folder (same layout as `implementation.ipynb`).

## 1. Imports and reproducibility

In [1]:
import os
import re
import time
import random
import pickle
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cpu


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Config
Same as the base notebook, plus new hyperparameters for regularization, LR scheduling, and early stopping.

In [2]:
# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA_DIR = "liar2"          # folder containing train.csv, valid.csv, test.csv
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
VALID_CSV = os.path.join(DATA_DIR, "valid.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")

CHECKPOINT_DIR = "checkpoints_v2"          # separate from the base run's checkpoints
VOCAB_PATH = os.path.join(CHECKPOINT_DIR, "vocabs.pkl")
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "fdhn_best_v2.pt")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Dataset / label settings
# ---------------------------------------------------------------------------
NUM_CLASSES = 6
LABEL_NAMES = ["pants-fire", "false", "barely-true", "half-true", "mostly-true", "true"]

NUMERICAL_COLUMNS = [
    "true_counts", "mostly_true_counts", "half_true_counts",
    "mostly_false_counts", "false_counts", "pants_on_fire_counts",
]
CONTEXT_COLUMNS = ["date", "subject", "speaker", "speaker_description", "state_info", "context"]

STATEMENT_COLUMN = "statement"
JUSTIFICATION_COLUMN = "justification"
LABEL_COLUMN = "label"

# ---------------------------------------------------------------------------
# Tokenization / sequence lengths
# ---------------------------------------------------------------------------
MAX_STATEMENT_LEN = 40
MAX_CONTEXT_LEN = 60
MAX_JUSTIFICATION_LEN = 120
MIN_WORD_FREQ = 2

PAD_TOKEN, PAD_IDX = "<PAD>", 0
UNK_TOKEN, UNK_IDX = "<UNK>", 1

# ---------------------------------------------------------------------------
# Model hyperparameters (Section V-A)
# ---------------------------------------------------------------------------
EMBED_DIM = 128
KERNEL_SIZES = (3, 4, 5)
NUM_FILTERS = 128
DROPOUT = 0.5

NUMERICAL_FEATURE_DIM = 64
CNN_OUT_CHANNELS = 32
LSTM_HIDDEN = 64

USE_JUSTIFICATION = True

# ---------------------------------------------------------------------------
# Training hyperparameters (base)
# ---------------------------------------------------------------------------
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
SEED = 42

# ---------------------------------------------------------------------------
# NEW: regularization / scheduling / early stopping
# ---------------------------------------------------------------------------
WEIGHT_DECAY = 1e-4        # L2 regularization via AdamW
GRAD_CLIP_NORM = 1.0       # gradient clipping threshold
LR_PATIENCE = 2            # epochs to wait before reducing LR
LR_FACTOR = 0.5            # multiply LR by this when val loss plateaus
EARLY_STOP_PATIENCE = 4    # stop training after this many epochs with no val improvement
NUM_EPOCHS = 20            # allow more epochs since early stopping will cut it short

## 3. Preprocessing functions

In [3]:
def simple_tokenize(text):
    """Lowercase, keep only alphanumerics, split on whitespace."""
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()


def build_vocab(token_lists, min_freq=MIN_WORD_FREQ):
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)
    vocab = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab


def encode_tokens(tokens, vocab, max_len):
    ids = [vocab.get(t, UNK_IDX) for t in tokens[:max_len]]
    if len(ids) < max_len:
        ids = ids + [PAD_IDX] * (max_len - len(ids))
    return ids


def build_context_text(row):
    parts = []
    for col in CONTEXT_COLUMNS:
        val = row.get(col, "")
        if pd.notna(val):
            parts.append(str(val))
    return " ".join(parts)


def fit_vocabs_and_scaler(train_df):
    """Fit everything on the TRAINING split only, to avoid leakage."""
    statement_tokens = [simple_tokenize(s) for s in train_df[STATEMENT_COLUMN]]
    context_tokens = [simple_tokenize(build_context_text(row)) for _, row in train_df.iterrows()]
    justification_tokens = [
        simple_tokenize(s) for s in train_df.get(JUSTIFICATION_COLUMN, pd.Series([""] * len(train_df)))
    ]

    statement_vocab = build_vocab(statement_tokens)
    context_vocab = build_vocab(context_tokens)
    justification_vocab = build_vocab(justification_tokens)

    numerical = train_df[NUMERICAL_COLUMNS].fillna(0).astype(float).values
    mean = numerical.mean(axis=0)
    std = numerical.std(axis=0)
    std[std == 0] = 1.0

    return {
        "statement_vocab": statement_vocab,
        "context_vocab": context_vocab,
        "justification_vocab": justification_vocab,
        "numerical_mean": mean,
        "numerical_std": std,
    }


def encode_row(row, vocabs):
    """Encode a single dataframe row into model-ready arrays."""
    statement_ids = encode_tokens(simple_tokenize(row[STATEMENT_COLUMN]), vocabs["statement_vocab"], MAX_STATEMENT_LEN)
    context_ids = encode_tokens(simple_tokenize(build_context_text(row)), vocabs["context_vocab"], MAX_CONTEXT_LEN)
    justification_ids = encode_tokens(
        simple_tokenize(row.get(JUSTIFICATION_COLUMN, "")), vocabs["justification_vocab"], MAX_JUSTIFICATION_LEN
    )

    raw_numerical = np.array([float(row.get(c, 0) or 0) for c in NUMERICAL_COLUMNS], dtype=np.float32)
    numerical = (raw_numerical - vocabs["numerical_mean"]) / vocabs["numerical_std"]

    label = int(row[LABEL_COLUMN])

    return {
        "statement": np.array(statement_ids, dtype=np.int64),
        "context": np.array(context_ids, dtype=np.int64),
        "justification": np.array(justification_ids, dtype=np.int64),
        "numerical": numerical.astype(np.float32),
        "label": label,
    }

## 4. Load training data and build vocabularies

In [4]:
print(f"Loading training data from {TRAIN_CSV} ...")
train_df = pd.read_csv(TRAIN_CSV)
print(f"  {len(train_df)} rows")

print("Fitting vocabularies and scaler on training split ...")
vocabs = fit_vocabs_and_scaler(train_df)
print(f"  statement vocab size:     {len(vocabs['statement_vocab'])}")
print(f"  context vocab size:       {len(vocabs['context_vocab'])}")
print(f"  justification vocab size: {len(vocabs['justification_vocab'])}")

with open(VOCAB_PATH, "wb") as f:
    pickle.dump(vocabs, f)
print(f"Saved vocabularies to {VOCAB_PATH}")

Loading training data from liar2/train.csv ...
  18369 rows
Fitting vocabularies and scaler on training split ...
  statement vocab size:     10046
  context vocab size:       8226
  justification vocab size: 21756
Saved vocabularies to checkpoints_v2/vocabs.pkl


## 5. PyTorch Dataset / DataLoaders

In [5]:
class LiarDataset(Dataset):
    def __init__(self, csv_path, vocabs):
        self.df = pd.read_csv(csv_path)
        self.vocabs = vocabs

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoded = encode_row(row, self.vocabs)
        return {
            "statement": torch.tensor(encoded["statement"], dtype=torch.long),
            "context": torch.tensor(encoded["context"], dtype=torch.long),
            "justification": torch.tensor(encoded["justification"], dtype=torch.long),
            "numerical": torch.tensor(encoded["numerical"], dtype=torch.float32),
            "label": torch.tensor(encoded["label"], dtype=torch.long),
        }


train_ds = LiarDataset(TRAIN_CSV, vocabs)
valid_ds = LiarDataset(VALID_CSV, vocabs)
test_ds = LiarDataset(TEST_CSV, vocabs)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)} | Valid batches: {len(valid_loader)} | Test batches: {len(test_loader)}")

Train batches: 288 | Valid batches: 36 | Test batches: 36


## 6. NEW: Class weights
Computes inverse-frequency weights so the loss penalizes mistakes on rarer classes (e.g. `pants-fire`, `true`) more heavily — directly targets the weak recall seen in the base run.

In [6]:
label_counts = train_df["label"].value_counts().sort_index()
print("Training label counts:\n", label_counts)

class_counts = label_counts.values.astype(float)
class_weights = class_counts.sum() / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print("\nClass weights (higher = rarer class, weighted more in loss):")
for name, w in zip(LABEL_NAMES, class_weights.cpu().numpy()):
    print(f"  {name:>12s}: {w:.3f}")

Training label counts:
 label
0    2425
1    5284
2    2882
3    2967
4    2743
5    2068
Name: count, dtype: int64

Class weights (higher = rarer class, weighted more in loss):
    pants-fire: 1.262
         false: 0.579
   barely-true: 1.062
     half-true: 1.032
   mostly-true: 1.116
          true: 1.480


## 7. Model architecture (unchanged from base — Section IV of the paper)

In [7]:
class TextCNN(nn.Module):
    """Section IV-A / IV-B: TextCNN with 3 parallel conv kernel sizes + max-pool."""

    def __init__(self, vocab_size, embed_dim, kernel_sizes, num_filters, output_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.convs = nn.ModuleList(
            [nn.Conv1d(embed_dim, num_filters, kernel_size=k) for k in kernel_sizes]
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), output_dim)

    def forward(self, x):
        emb = self.embedding(x).permute(0, 2, 1)
        pooled_outputs = []
        for conv in self.convs:
            c = F.relu(conv(emb))
            p = F.max_pool1d(c, kernel_size=c.size(2)).squeeze(2)
            pooled_outputs.append(p)
        cat = torch.cat(pooled_outputs, dim=1)
        cat = self.dropout(cat)
        return self.fc(cat)


class CNNBiLSTM(nn.Module):
    """Section IV-C: linear+dropout -> CNN(out=32, kernel=1) -> Bi-LSTM -> linear."""

    def __init__(self, num_numerical_features, feature_dim, cnn_out_channels, lstm_hidden, output_dim, dropout):
        super().__init__()
        self.input_linear = nn.Linear(num_numerical_features, feature_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.cnn = nn.Conv1d(in_channels=1, out_channels=cnn_out_channels, kernel_size=1)
        self.lstm = nn.LSTM(
            input_size=cnn_out_channels, hidden_size=lstm_hidden,
            num_layers=1, batch_first=True, bidirectional=True,
        )
        self.dropout2 = nn.Dropout(dropout)
        self.output_linear = nn.Linear(lstm_hidden * 2, output_dim)
        self.hidden_dim = lstm_hidden * 2

    def forward(self, x):
        h = self.input_linear(x)
        h = self.dropout1(h)
        h = h.unsqueeze(1)
        h = F.relu(self.cnn(h))
        h = h.permute(0, 2, 1)
        lstm_out, _ = self.lstm(h)
        pooled = lstm_out[:, -1, :]
        pooled = self.dropout2(pooled)
        out = self.output_linear(pooled)
        return out, pooled


class FuzzyLayer(nn.Module):
    """Section IV-D: Gaussian-membership Fuzzy Layer (Eq. 1 and Eq. 2)."""

    def __init__(self, feature_dim, num_classes):
        super().__init__()
        self.feature_dim = feature_dim
        self.num_classes = num_classes
        self.centers = nn.Parameter(torch.randn(num_classes, feature_dim) * 0.1)
        self.log_sigmas = nn.Parameter(torch.zeros(num_classes, feature_dim))

    def forward(self, x):
        sigmas = F.softplus(self.log_sigmas) + 1e-3
        x = x.unsqueeze(1)
        centers = self.centers.unsqueeze(0)
        sigmas = sigmas.unsqueeze(0)
        membership = torch.exp(-0.5 * ((x - centers) / sigmas) ** 2)
        mu = membership.mean(dim=2)
        return mu


class FDHN(nn.Module):
    """Full Fuzzy Deep Hybrid Network (Fig. 1)."""

    def __init__(self, statement_vocab_size, context_vocab_size,
                 justification_vocab_size=None, use_justification=USE_JUSTIFICATION):
        super().__init__()
        self.use_justification = use_justification and justification_vocab_size is not None

        self.news_text_module = TextCNN(
            statement_vocab_size, EMBED_DIM, KERNEL_SIZES, NUM_FILTERS, NUM_CLASSES, DROPOUT
        )
        self.textual_context_module = TextCNN(
            context_vocab_size, EMBED_DIM, KERNEL_SIZES, NUM_FILTERS, NUM_CLASSES, DROPOUT
        )
        self.numerical_context_module = CNNBiLSTM(
            num_numerical_features=len(NUMERICAL_COLUMNS),
            feature_dim=NUMERICAL_FEATURE_DIM,
            cnn_out_channels=CNN_OUT_CHANNELS,
            lstm_hidden=LSTM_HIDDEN,
            output_dim=NUM_CLASSES,
            dropout=DROPOUT,
        )
        self.fuzzy_layer = FuzzyLayer(
            feature_dim=self.numerical_context_module.hidden_dim, num_classes=NUM_CLASSES
        )

        num_branches = 4
        if self.use_justification:
            self.justification_module = TextCNN(
                justification_vocab_size, EMBED_DIM, KERNEL_SIZES, NUM_FILTERS, NUM_CLASSES, DROPOUT
            )
            num_branches = 5

        self.fuse_layer = nn.Linear(NUM_CLASSES * num_branches, NUM_CLASSES)

    def forward(self, statement, context, numerical, justification=None):
        out1 = self.news_text_module(statement)
        out2 = self.textual_context_module(context)
        out3, numerical_hidden = self.numerical_context_module(numerical)
        out4 = self.fuzzy_layer(numerical_hidden)

        outputs = [out1, out2, out3, out4]

        if self.use_justification:
            assert justification is not None, "justification tensor required when use_justification=True"
            out5 = self.justification_module(justification)
            outputs.append(out5)

        fused = torch.cat(outputs, dim=1)
        logits = self.fuse_layer(fused)
        return logits

## 8. Instantiate a fresh model

In [8]:
model = FDHN(
    statement_vocab_size=len(vocabs["statement_vocab"]),
    context_vocab_size=len(vocabs["context_vocab"]),
    justification_vocab_size=len(vocabs["justification_vocab"]),
    use_justification=USE_JUSTIFICATION,
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {num_params:,}")

Trainable parameters: 5,774,674


## 9. NEW: run_epoch with gradient clipping
Same as base, plus an optional `clip_norm` argument.

In [9]:
def run_epoch(model, loader, criterion, optimizer=None, device=DEVICE, clip_norm=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds, all_labels = [], []

    context_manager = torch.enable_grad() if is_train else torch.no_grad()
    with context_manager:
        for batch in tqdm(loader, leave=False):
            statement = batch["statement"].to(device)
            context = batch["context"].to(device)
            justification = batch["justification"].to(device)
            numerical = batch["numerical"].to(device)
            labels = batch["label"].to(device)

            if is_train:
                optimizer.zero_grad()

            logits = model(statement, context, numerical, justification)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                if clip_norm is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1).detach().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    f1_micro = f1_score(all_labels, all_preds, average="micro")
    return avg_loss, accuracy, f1_macro, f1_micro

## 10. NEW: Improved training loop
AdamW + weight decay, class-weighted loss, gradient clipping, `ReduceLROnPlateau`, and early stopping.

In [10]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=LR_FACTOR, patience=LR_PATIENCE
)

best_val_acc = 0.0
epochs_without_improvement = 0

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()
    train_loss, train_acc, train_f1_macro, train_f1_micro = run_epoch(
        model, train_loader, criterion, optimizer, clip_norm=GRAD_CLIP_NORM
    )
    val_loss, val_acc, val_f1_macro, val_f1_micro = run_epoch(
        model, valid_loader, criterion, optimizer=None
    )
    elapsed = time.time() - start

    current_lr = optimizer.param_groups[0]["lr"]
    print(
        f"Epoch {epoch:2d}/{NUM_EPOCHS} | {elapsed:5.1f}s | lr={current_lr:.2e} | "
        f"train_loss={train_loss:.4f} acc={train_acc:.4f} f1m={train_f1_macro:.4f} | "
        f"val_loss={val_loss:.4f} acc={val_acc:.4f} f1_macro={val_f1_macro:.4f} f1_micro={val_f1_micro:.4f}"
    )

    scheduler.step(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_without_improvement = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  -> new best model saved (val_acc={val_acc:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  -> no improvement for {epochs_without_improvement} epoch(s)")

    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping triggered after epoch {epoch} "
              f"(no val improvement for {EARLY_STOP_PATIENCE} epochs).")
        break

print(f"\nBest validation accuracy: {best_val_acc:.4f}")

Epoch  1/20 |  79.2s | lr=1.00e-03 | train_loss=1.4830 acc=0.3654 f1m=0.3643 | val_loss=1.2886 acc=0.4615 f1_macro=0.4613 f1_micro=0.4615
  -> new best model saved (val_acc=0.4615)


Epoch  2/20 |  79.4s | lr=1.00e-03 | train_loss=1.2385 acc=0.4762 f1m=0.4765 | val_loss=1.2035 acc=0.4802 f1_macro=0.4780 f1_micro=0.4802
  -> new best model saved (val_acc=0.4802)


Epoch  3/20 |  81.7s | lr=1.00e-03 | train_loss=1.1123 acc=0.5337 f1m=0.5345 | val_loss=1.1625 acc=0.5346 f1_macro=0.5230 f1_micro=0.5346
  -> new best model saved (val_acc=0.5346)


Epoch  4/20 |  83.6s | lr=1.00e-03 | train_loss=1.0005 acc=0.5797 f1m=0.5816 | val_loss=1.1461 acc=0.5337 f1_macro=0.5237 f1_micro=0.5337
  -> no improvement for 1 epoch(s)


Epoch  5/20 |  90.4s | lr=1.00e-03 | train_loss=0.8859 acc=0.6316 f1m=0.6350 | val_loss=1.1813 acc=0.5355 f1_macro=0.5287 f1_micro=0.5355
  -> new best model saved (val_acc=0.5355)


...

Epoch  7/20 |  88.3s | lr=1.00e-03 | train_loss=0.6637 acc=0.7292 f1m=0.7335 | val_loss=1.2512 acc=0.5420 f1_macro=0.5320 f1_micro=0.5420
  -> new best model saved (val_acc=0.5420)


Epoch  8/20 |  89.5s | lr=5.00e-04 | train_loss=0.5120 acc=0.7929 f1m=0.7982 | val_loss=1.3252 acc=0.5398 f1_macro=0.5283 f1_micro=0.5398
  -> no improvement for 1 epoch(s)


Epoch  9/20 |  90.1s | lr=5.00e-04 | train_loss=0.4168 acc=0.8281 f1m=0.8332 | val_loss=1.4049 acc=0.5403 f1_macro=0.5299 f1_micro=0.5403
  -> no improvement for 2 epoch(s)


Epoch 10/20 |  90.4s | lr=5.00e-04 | train_loss=0.3494 acc=0.8611 f1m=0.8654 | val_loss=1.4670 acc=0.5320 f1_macro=0.5253 f1_micro=0.5320
  -> no improvement for 3 epoch(s)


Epoch 11/20 |  93.4s | lr=2.50e-04 | train_loss=0.2812 acc=0.8892 f1m=0.8926 | val_loss=1.4832 acc=0.5411 f1_macro=0.5330 f1_micro=0.5411
  -> no improvement for 4 epoch(s)

Early stopping triggered after epoch 11 (no val improvement for 4 epochs).

Best validation accuracy: 0.5420


## 11. Evaluate on the test set

In [11]:
print("Evaluating best model on the TEST set ...")
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
test_loss, test_acc, test_f1_macro, test_f1_micro = run_epoch(model, test_loader, criterion, optimizer=None)

print(f"TEST  loss={test_loss:.4f} accuracy={test_acc:.4f} f1_macro={test_f1_macro:.4f} f1_micro={test_f1_micro:.4f}")
print()
print("Base run (no regularization):        accuracy=0.548  f1_macro=0.519")
print("Paper's reference (Table 7, LIAR2 Mix): accuracy=0.702  f1_macro=0.696")

Evaluating best model on the TEST set ...


TEST  loss=1.2312 accuracy=0.5401 f1_macro=0.5274 f1_micro=0.5401

Base run (no regularization):        accuracy=0.548  f1_macro=0.519
Paper's reference (Table 7, LIAR2 Mix): accuracy=0.702  f1_macro=0.696


## 12. Detailed evaluation: classification report + confusion matrix

In [12]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        statement = batch["statement"].to(DEVICE)
        context = batch["context"].to(DEVICE)
        justification = batch["justification"].to(DEVICE)
        numerical = batch["numerical"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        logits = model(statement, context, numerical, justification)
        preds = logits.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

print("Classification report:\n")
print(classification_report(all_labels, all_preds, target_names=LABEL_NAMES, digits=3))

print("Confusion matrix (rows=true, cols=predicted):")
print(confusion_matrix(all_labels, all_preds))

Classification report:

              precision    recall  f1-score   support

  pants-fire      0.562     0.492     0.525       303
       false      0.623     0.620     0.621       660
 barely-true      0.570     0.450     0.503       360
   half-true      0.525     0.447     0.483       371
 mostly-true      0.429     0.627     0.509       343
        true      0.509     0.537     0.523       259

    accuracy                          0.540      2296
   macro avg      0.536     0.529     0.527      2296
weighted avg      0.549     0.540     0.540      2296

Confusion matrix (rows=true, cols=predicted):
[[149 108  21   6   9  10]
 [ 91 409  39  34  46  41]
 [ 10  57 162  65  51  15]
 [  6  33  33 166 107  26]
 [  2  23  22  39 215  42]
 [  7  27   7   6  73 139]]


## 13. Prediction on new statements

In [ ]:
def predict_statement(statement, speaker="", subject="", speaker_description="",
                       state_info="national", context="", date="",
                       true_counts=0, mostly_true_counts=0, half_true_counts=0,
                       mostly_false_counts=0, false_counts=0, pants_on_fire_counts=0,
                       justification=""):
    row = {
        "statement": statement, "date": date, "subject": subject, "speaker": speaker,
        "speaker_description": speaker_description, "state_info": state_info, "context": context,
        "true_counts": true_counts, "mostly_true_counts": mostly_true_counts,
        "half_true_counts": half_true_counts, "mostly_false_counts": mostly_false_counts,
        "false_counts": false_counts, "pants_on_fire_counts": pants_on_fire_counts,
        "justification": justification, "label": 0,
    }
    encoded = encode_row(row, vocabs)
    statement_t = torch.tensor(encoded["statement"]).unsqueeze(0).to(DEVICE)
    context_t = torch.tensor(encoded["context"]).unsqueeze(0).to(DEVICE)
    justification_t = torch.tensor(encoded["justification"]).unsqueeze(0).to(DEVICE)
    numerical_t = torch.tensor(encoded["numerical"]).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(statement_t, context_t, numerical_t, justification_t)
        probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()

    predicted_idx = probs.argmax()
    print(f"\nPredicted: {LABEL_NAMES[predicted_idx]}  (confidence: {probs[predicted_idx]:.1%})\n")
    for name, p in sorted(zip(LABEL_NAMES, probs), key=lambda x: -x[1]):
        print(f"  {name:>12s}: {p:.1%}")


def predict_from_user_input():
    print("=" * 60)
    print("FDHN Fake News Detector v2 — Interactive Prediction")
    print("=" * 60)
    print("(Press Enter to skip any optional field and use its default)\n")

    statement = input("Statement / claim (required): ").strip()
    while not statement:
        statement = input("Statement cannot be empty. Please enter a claim: ").strip()

    speaker = input("Speaker (e.g. 'joe-biden') [optional]: ").strip()
    subject = input("Subject (e.g. 'economy,jobs') [optional]: ").strip()
    speaker_description = input("Speaker description [optional]: ").strip()
    state_info = input("State info [default: national]: ").strip() or "national"
    context = input("Context (e.g. 'a campaign speech') [optional]: ").strip()
    date = input("Date (e.g. '2020-01-01') [optional]: ").strip()
    justification = input("Justification / fact-checker rationale [optional]: ").strip()

    def get_int(prompt, default=0):
        val = input(f"{prompt} [default: {default}]: ").strip()
        return int(val) if val else default

    print("\n--- Speaker's PolitiFact history counts ---")
    true_counts = get_int("True counts")
    mostly_true_counts = get_int("Mostly-true counts")
    half_true_counts = get_int("Half-true counts")
    mostly_false_counts = get_int("Mostly-false (barely-true) counts")
    false_counts = get_int("False counts")
    pants_on_fire_counts = get_int("Pants-on-fire counts")

    predict_statement(
        statement=statement, speaker=speaker, subject=subject,
        speaker_description=speaker_description, state_info=state_info,
        context=context, date=date,
        true_counts=true_counts, mostly_true_counts=mostly_true_counts,
        half_true_counts=half_true_counts, mostly_false_counts=mostly_false_counts,
        false_counts=false_counts, pants_on_fire_counts=pants_on_fire_counts,
        justification=justification,
    )


# Run interactively:
predict_from_user_input()

FDHN Fake News Detector v2 — Interactive Prediction
(Press Enter to skip any optional field and use its default)



## 14. Example predictions
A few pre-filled examples you can run without typing anything.

In [ ]:
examples = [
    {
        "name": "Goodyear political hats claim",
        "statement": "Goodyear Tires announced a ban on all political hats at their factories.",
        "speaker": "a-commentator",
        "subject": "politics,workplace",
        "context": "a social media post",
    },
    {
        "name": "Unemployment claim",
        "statement": "The unemployment rate has fallen to its lowest level in fifty years.",
        "speaker": "a-politician",
        "subject": "economy,jobs",
        "context": "a televised debate",
        "true_counts": 40, "mostly_true_counts": 30, "half_true_counts": 10,
        "mostly_false_counts": 5, "false_counts": 3, "pants_on_fire_counts": 1,
    },
    {
        "name": "Chocolate and memory claim",
        "statement": "Daily chocolate consumption improves memory.",
        "subject": "health,science",
        "context": "an online article",
    },
]

for example in examples:
    print(f"\n{'=' * 60}\n{example['name']}\n{'=' * 60}")
    prediction_input = {key: value for key, value in example.items() if key != "name"}
    predict_statement(**prediction_input)

## Summary of changes vs. base implementation

| Change | Why |
|---|---|
| `AdamW` + `weight_decay=1e-4` | L2 regularization to reduce memorization |
| Class-weighted `CrossEntropyLoss` | Targets weak recall on rarer classes (`pants-fire`, `true`) |
| Gradient clipping (`clip_norm=1.0`) | Stabilizes training |
| `ReduceLROnPlateau` scheduler | Shrinks LR once validation loss stalls, instead of a fixed LR for all 10 epochs |
| Early stopping (`patience=4`) | Stops before the model overfits past its best validation point |
| `NUM_EPOCHS=20` | Safe to raise since early stopping will cut training short automatically |

Compare this notebook's Section 11/12 output directly against your base `implementation.ipynb`'s equivalent cells.